# 04. Prior-design geographic prediction with released-by-February-2020 inputs

Run revised 01--03 first. This notebook evaluates every planned primary algorithm separately and exports metro/micro scores.
The revised final split is not an untouched confirmation set: earlier results used these same CBSAs.
All computations are delegated to the readable design_models.py module. See RERUN_DESIGN_2020.md before execution.


The core suite uses the revised coverage-first feature set. Stage 2 records observed coverage before imputation and retains the 80% threshold within each training fold.


## C46 forecast sensitivity, outliers and growth

Run the corresponding long-horizon cell in notebook 01 first. This cell uses notebook
02 predictors and recomputes the outcome for March 2020 through December 2023 on
the same CBSAs under every forecast rule. It also includes the existing Stage 1
selected, damped and ensemble paths when those outputs are available. The remaining
cells run the prior geographic validation.

The audit compares C46 by model family, identifies large outcomes and held-division
residuals, and matches names to the original September 2018 OMB workbook.
C46 is the sum of positive monthly shortfalls in percentage-point-months. Dividing
by 46 gives the mean monthly positive shortfall. The signed sum retains surpluses.

The recent Q2 growth check compares its association with actual growth, projected
growth, and the change in C46 relative to seasonal naive while holding observed
employment fixed. Historical placebo outcomes provide a second check. A fixed
ridge model with preprocessing fitted in each training fold measures whether adding
recent growth improves held-division prediction on each outcome. The prediction
comparison is descriptive and does not identify the effect of any local shock.


In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

c46_start = Path(os.environ.get("RL_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
c46_root = next((p for p in [c46_start, *c46_start.parents]
                 if (p / "data/processed/omb2018_geography_manifest.json").is_file()), None)
if c46_root is None:
    raise FileNotFoundError("Open the practicum project or set RL_PROJECT_ROOT.")
c46_code = Path(os.environ["RL_C46_CODE"]).resolve() if "RL_C46_CODE" in os.environ else c46_root / "code"
if str(c46_code) not in sys.path:
    sys.path.insert(0, str(c46_code))
c46_output = os.environ.get("RL_C46_OUTPUT")

from c46_forecast_validity import run_stage2

from stage1_primary import require_primary
require_primary(c46_root, require_stage2=True)

c46_dir = run_stage2(c46_root, c46_output)
for c46_name in ("c46_by_scenario.csv", "c46_by_model_family.csv", "named_cbsa_verification.csv",
                 "growth_mechanical_association.csv", "geographic_growth_ablation.csv"):
    print(c46_name)
    display(pd.read_csv(c46_dir / c46_name, dtype={"cbsa_code": "string"}))
print("Full results, including historical placebos and residuals, saved to", c46_dir)


c46_by_scenario.csv


,scenario,n,median_c46,p90_c46,max_c46,max_mean_monthly_shortfall_pp
0,damped_log_seasonal_trend,936,154.306717,355.056449,1230.525857,26.750562
1,existing_stage1,936,154.164841,347.265921,1158.772324,25.190703
2,existing_stage1_damped,936,154.306717,355.056449,1230.525857,26.750562
3,existing_stage1_ensemble,936,150.457478,343.841105,996.997407,21.673857
4,fixed_ensemble,936,154.164841,347.265921,1158.772324,25.190703
5,seasonal_naive,936,119.266909,320.389743,967.387327,21.030159
6,selected_12,936,188.902465,393.967341,1141.376000,24.812522
7,selected_24,936,192.137278,399.574327,1141.376000,24.812522
8,selected_36,936,190.841540,402.173181,1141.376000,24.812522
9,selected_48,936,190.762115,396.141573,1149.251369,24.983725


c46_by_model_family.csv


,scenario,family,n,mean_c46,median_c46,p90_c46,max_c46,median_projected_growth,naive_fallbacks
0,damped_log_seasonal_trend,damped_log_seasonal_trend,936,184.516009,154.306717,355.056449,1230.525857,0.011018,3
1,existing_stage1,fixed_ensemble,936,181.252044,154.164841,347.265921,1158.772324,0.013823,0
2,existing_stage1_damped,fixed_ensemble,936,184.516009,154.306717,355.056449,1230.525857,0.011018,0
3,existing_stage1_ensemble,fixed_ensemble,936,176.773840,150.457478,343.841105,996.997407,0.006857,0
4,fixed_ensemble,fixed_ensemble,936,181.252044,154.164841,347.265921,1158.772324,0.013823,0
5,seasonal_naive,seasonal_naive,936,151.981906,119.266909,320.389743,967.387327,0.000000,0
6,selected_12,damped_log_seasonal_trend,151,185.945953,142.995215,377.196381,834.147577,0.009173,0
7,selected_12,log_seasonal_trend,188,244.585604,232.188116,458.203921,706.565841,0.041996,0
8,selected_12,seasonal_drift,317,228.753620,218.289193,385.686034,1141.376000,0.050913,0
9,selected_12,seasonal_naive,280,181.767666,153.853526,377.177501,967.387327,0.000000,0


named_cbsa_verification.csv


,cbsa_code,cbsa_name,family,scenario,c46,mean_monthly_shortfall_pp,qcew2018to2019q2_growth,projected_growth,actual_growth,c46_minus_naive,signed46_minus_naive,naive_fallback,predicted,abs_error,residual_rank,c46_rank,in_analysis_sample,pre_cutoff_complete_months,pre_cutoff_required_months,archived_c46_residual_rank
0,16100,"Carlsbad-Artesia, NM",damped_log_seasonal_trend,damped_log_seasonal_trend,364.867918,7.931911,0.094174,0.011134,-0.062256,39.509304,48.068038,False,239.647011,125.220907,249.0,85.0,True,114,114,1.0
1,16100,"Carlsbad-Artesia, NM",fixed_ensemble,existing_stage1,461.900297,10.041311,0.094174,0.043311,-0.062256,136.541684,148.712768,False,245.185232,216.715065,66.0,31.0,True,114,114,1.0
2,16100,"Carlsbad-Artesia, NM",fixed_ensemble,existing_stage1_damped,364.867918,7.931911,0.094174,0.011134,-0.062256,39.509304,48.068038,False,239.647011,125.220907,249.0,85.0,True,114,114,1.0
3,16100,"Carlsbad-Artesia, NM",fixed_ensemble,existing_stage1_ensemble,343.937292,7.476898,0.094174,0.005567,-0.062256,18.578679,24.519172,False,211.331650,132.605643,216.0,94.0,True,114,114,1.0
4,16100,"Carlsbad-Artesia, NM",fixed_ensemble,fixed_ensemble,461.900297,10.041311,0.094174,0.043311,-0.062256,136.541684,148.712768,False,245.185232,216.715065,66.0,31.0,True,114,114,1.0
5,16100,"Carlsbad-Artesia, NM",seasonal_naive,seasonal_naive,325.358613,7.073013,0.094174,0.000000,-0.062256,0.000000,0.000000,False,152.090112,173.268501,98.0,93.0,True,114,114,1.0
6,16100,"Carlsbad-Artesia, NM",log_seasonal_trend,selected_12,473.779227,10.299548,0.094174,0.041058,-0.062256,148.420613,155.846731,False,269.606076,204.173150,98.0,45.0,True,114,114,1.0
7,16100,"Carlsbad-Artesia, NM",seasonal_naive,selected_24,325.358613,7.073013,0.094174,0.000000,-0.062256,0.000000,0.000000,False,274.537901,50.820712,662.0,189.0,True,114,114,1.0
8,16100,"Carlsbad-Artesia, NM",seasonal_naive,selected_36,325.358613,7.073013,0.094174,0.000000,-0.062256,0.000000,0.000000,False,274.054236,51.304377,657.0,197.0,True,114,114,1.0
9,16100,"Carlsbad-Artesia, NM",damped_log_seasonal_trend,selected_48,364.867918,7.931911,0.094174,0.011134,-0.062256,39.509304,48.068038,False,287.921150,76.946768,529.0,146.0,True,114,114,1.0


growth_mechanical_association.csv


,scenario,metric,n,pearson,spearman
0,damped_log_seasonal_trend,projected_growth,936,0.106104,0.153189
1,damped_log_seasonal_trend,actual_growth,936,0.314705,0.341157
2,damped_log_seasonal_trend,c46,936,-0.269018,-0.322629
3,damped_log_seasonal_trend,signed46,936,-0.348857,-0.355453
4,damped_log_seasonal_trend,c46_minus_naive,936,-0.002958,-0.011306
...,...,...,...,...,...
61,selected_54,actual_growth,936,0.314705,0.341157
62,selected_54,c46,936,-0.179615,-0.174163
63,selected_54,signed46,936,-0.238655,-0.181398
64,selected_54,c46_minus_naive,936,0.108417,0.146436


geographic_growth_ablation.csv


,scenario,feature_mode,n,mae,rmse,r2,oos_r2_vs_training_mean,baseline_median_mae,skill_vs_median_pct,bias
0,damped_log_seasonal_trend,existing_primary,936,95.901974,133.458384,0.125714,0.148033,108.540414,11.643995,-1.279064
1,damped_log_seasonal_trend,no_growth,936,100.196275,136.777637,0.081684,0.105128,108.540414,7.687587,-0.736194
2,damped_log_seasonal_trend,primary_plus_recent,936,92.203648,128.948533,0.183804,0.204640,108.540414,15.051321,-0.259794
3,damped_log_seasonal_trend,recent_growth,936,94.356943,130.962200,0.158113,0.179605,108.540414,13.067457,0.287502
4,existing_stage1,existing_primary,936,96.100961,132.940178,0.112017,0.135023,106.868013,10.075093,-0.758088
5,existing_stage1,no_growth,936,99.114784,135.568107,0.076564,0.100488,106.868013,7.254957,-0.444454
6,existing_stage1,primary_plus_recent,936,93.455974,130.003619,0.150814,0.172814,106.868013,12.550096,0.080334
7,existing_stage1,recent_growth,936,95.336724,131.600619,0.129823,0.152367,106.868013,10.790215,0.414396
8,existing_stage1_damped,existing_primary,936,95.901974,133.458384,0.125714,0.148033,108.540414,11.643995,-1.279064
9,existing_stage1_damped,no_growth,936,100.196275,136.777637,0.081684,0.105128,108.540414,7.687587,-0.736194


Full results, including historical placebos and residuals, saved to C:\Users\adamt\OneDrive\Documents\Directed Practicum\regional-labor-resilience\output\c46_forecast_validity


In [2]:
import os
os.environ.setdefault("RL_N_JOBS", "-1")

'-1'

In [3]:
from pathlib import Path
import os,sys
ROOT=Path(os.environ['RL_PROJECT_ROOT']).resolve() if os.environ.get('RL_PROJECT_ROOT') else next(
    (p for p in [Path.cwd(),*Path.cwd().parents] if (p/'code'/'design_models.py').exists()),None)
if ROOT is None: raise FileNotFoundError('Open the project root in VS Code or set RL_PROJECT_ROOT')
if str(ROOT/'code') not in sys.path: sys.path.insert(0,str(ROOT/'code'))
from stage1_primary import require_primary
require_primary(ROOT, require_stage2=True)
from design_models import run
predictions=run(ROOT)
print('Finished. Principal tables: output/design2020_scores.csv and output/design2020_group_scores.csv')


Primary feature coverage (actual exclusion is learned inside each fold):
                                                            feature  coverage
                           qcew2019q2_private_goods_producing_share  0.987179
                               qcew2019q2_private_share_all_covered  0.994658
                                       acs2018_bachelors_plus_share  0.998932
                        acs2018_log_grouped_median_household_income  0.998932
                            acs2018_labor_force_participation_share  0.998932
                                        laus2018_unemployment_share  0.993590
                               acs2018_broadband_subscription_share  0.998932
                                              acs2018_poverty_share  0.998932
                               qcew2015to2018_annualized_log_growth  1.000000
                                     log_qcew2018_annual_employment  1.000000
                                                    is_micropolitan  